# **Imports and functions**

In [6]:
from PIL import Image, ImageEnhance
import glob
import os
import pandas as pd
img_folder = r"I:\Whole_brains\M2_1\Stitched"



In [2]:


def convert_tiff_to_24bit_png(input_tiff_path, output_png_path, resize, factor_resize, brightness, brightness_factor, color , noDAPI : bool = False):
    """
    Converts a TIFF image to a 24-bit PNG image.

    Args:
        input_tiff_path (str): The path to the input TIFF file.
        output_png_path (str): The path to save the output PNG file.
    """
    try:
        # Open the TIFF image
        tiff_image = Image.open(input_tiff_path)

        # Generate color image by merging layers of TIFF file
        if color == True:
            tiff_image.seek(0)
            r = tiff_image.convert("L")
            
            tiff_image.seek(1)
            g = tiff_image.convert("L")
            
            tiff_image.seek(2)
            if noDAPI == True:
                b = Image.new("L", r.size, 0)
            else:
                b = tiff_image.convert("L")
            # Merge the individual grayscale frames into one RGB image
            rgb_image = Image.merge("RGB", (r, g, b))
        else:
            rgb_image = tiff_image.convert('RGB')  
        # Convert the image to 'RGB' mode for 24-bit color depth
        # This removes any alpha channel if present, resulting in a 24-bit image
        print(f'Converting {input_tiff_path} to (24-bit PNG).')
        
        
        if resize == True:
            print(f'Resizing by a factor: {factor_resize}')
            width, height = rgb_image.size
            rgb_image = rgb_image.resize((width // factor_resize, height // factor_resize))
            output_png_path = output_png_path + '_reduced'

        if brightness == True:
            enhancer = ImageEnhance.Brightness(rgb_image)
            rgb_image = enhancer.enhance(brightness_factor)


        # Save the image as a PNG
        rgb_image.save(f"{output_png_path}.png", format="PNG")
        print(f"Successfully converted '{input_tiff_path}' to '{output_png_path}' (24-bit PNG).")

    except FileNotFoundError:
        print(f"Error: The file '{input_tiff_path}' was not found.")
    except Exception as e:
        print(f"An error occurred during conversion: {e}")




def convert_tiff_to_24bit_png_multiple_chan(input_tiff_path, output_png_path, resize, factor_resize, brightness, brightness_factor, color , noDAPI : bool = False, channel : str = 'CH3'):
    """
    Converts a TIFF image to a 24-bit PNG image.

    Args:
        input_tiff_path (str): The path to the input TIFF file.
        output_png_path (str): The path to save the output PNG file.
        resize (bool): Wether image needs resizing or not
        factor_resize (int) : The factor by which the image should be resized (new img = 1/ factor * img)
        brightness (bool) : Wether image needs brightness adjustment or not
        brightness_factor (int) : The factor by which the brightness of the image should be multiplied (new img = factor * img)
        color (bool) : Wether the user wants a RGB color png output
        noDAPI (bool) : Remove DAPI channel from the RGB png output or not
    """
    try:
        # Open the TIFF image
        tiff_image = Image.open(input_tiff_path)

        # Generate color image by merging layers of TIFF file
        if color == True:
            tiff_image.seek(0)
            r = tiff_image.convert("L")
            
            tiff_image.seek(1)
            g = tiff_image.convert("L")
            
            tiff_image.seek(2)
            if noDAPI == True:
                b = Image.new("L", r.size, 0)
            else:
                b = tiff_image.convert("L")
            # Merge the individual grayscale frames into one RGB image
            rgb_image = Image.merge("RGB", (r, g, b))
        else:

            if channel == 'CH1':
                tiff_image.seek(2)
                rgb_image = tiff_image.convert('RGB')
            elif channel == 'CH2':
                tiff_image.seek(1)
                rgb_image = tiff_image.convert('RGB') 
            elif channel == 'CH3':
                tiff_image.seek(0)
                rgb_image = tiff_image.convert('RGB')
        # Convert the image to 'RGB' mode for 24-bit color depth
        # This removes any alpha channel if present, resulting in a 24-bit image
        print(f'Converting {input_tiff_path} to (24-bit PNG).')
        
        
        if resize == True:
            print(f'Resizing by a factor: {factor_resize}')
            width, height = rgb_image.size
            rgb_image = rgb_image.resize((width // factor_resize, height // factor_resize))
            output_png_path = output_png_path + '_reduced'

        if brightness == True:
            enhancer = ImageEnhance.Brightness(rgb_image)
            rgb_image = enhancer.enhance(brightness_factor)


        # Save the image as a PNG
        rgb_image.save(f"{output_png_path}.png", format="PNG")
        print(f"Successfully converted '{input_tiff_path}' to '{output_png_path}' (24-bit PNG).")

    except FileNotFoundError:
        print(f"Error: The file '{input_tiff_path}' was not found.")
    except Exception as e:
        print(f"An error occurred during conversion: {e}")

# **Resizing/Brightening/Converting images**

## For Single channel

In [ ]:
# Create arborescence
ilastik_output = os.path.join(img_folder,'images_24bits_ilastik')
quickNII_output = os.path.join(img_folder,'images_24bits_QuickNII')

if not os.path.isdir(ilastik_output):
    os.mkdir(ilastik_output)

if not os.path.isdir(quickNII_output):
    os.mkdir(quickNII_output)


file_list = [file for file in os.listdir(img_folder) if file.endswith(".tif")]

for filename in file_list:
    # Convert to png with a 0.2 resize for QuickNII
    # Up the brightness to better identify damages and edges
    convert_tiff_to_24bit_png(input_tiff_path= os.path.join(img_folder,filename), 
                              output_png_path= os.path.join(quickNII_output,f'{filename.split(sep =".")[0]}'),
                              resize = True,
                              factor_resize= 20,
                              brightness = True,
                              brightness_factor= 3
                              color = False)
    
    #Convert to png with a 0.1 resize for Ilastik
    convert_tiff_to_24bit_png(input_tiff_path= os.path.join(img_folder,filename), 
                              output_png_path= os.path.join(ilastik_output,f'{filename.split(sep =".")[0]}'),
                              resize = True,
                              factor_resize= 5,
                              brightness= False,
                              brightness_factor= None
                              color = False)

## For multiple channels

In [7]:
# List all channels folder
Channels = [channel for channel in os.listdir(img_folder) if os.path.isdir(os.path.join(img_folder, channel))]

quickNII_output = os.path.join(img_folder,'images_24bits_QuickNII')

if not os.path.isdir(quickNII_output):
    os.mkdir(quickNII_output)



######################################################################################################################### CREATE ONE GENERAL FOLDER FOR ALIGNMENT 
# Alignement slices based on DAPI
align_filelist = [file for file in os.listdir(os.path.join(img_folder, Channels[2])) if file.endswith(".tif")]

for filename in align_filelist:
    convert_tiff_to_24bit_png_multiple_chan(input_tiff_path= os.path.join(img_folder,Channels[2],filename), 
                                output_png_path= os.path.join(quickNII_output,f'{filename.split(sep =".")[0]}'),
                                resize = True,
                                factor_resize= 20,
                                brightness = True,
                                brightness_factor= 3,
                                color = False,
                                channel = Channels[2])

########################################################## CREATE ONE FOLDER PER CHANNEL FOR ILASTIK
for channel in Channels:

    # Create arborescence per channel
    ilastik_output = os.path.join(img_folder,channel,'images_24bits_ilastik')

    if not os.path.isdir(ilastik_output):
        os.mkdir(ilastik_output)



    file_list = [file for file in os.listdir(os.path.join(img_folder, channel)) if file.endswith(".tif")]

    for filename in file_list:
        if channel == 'Overlay':
            print("=================================== Creating color images for overlay channel ==================================")

            
            #Convert to png with a 0.1 resize for Ilastik
            convert_tiff_to_24bit_png_multiple_chan(input_tiff_path= os.path.join(img_folder,channel,filename), 
                                    output_png_path= os.path.join(ilastik_output,f'{filename.split(sep =".")[0]}'),
                                    resize = True,
                                    factor_resize= 5,
                                    brightness= False,
                                    brightness_factor= None,
                                    color = True,
                                    noDAPI = True)

        else:
            print(f"=================================== Creating grayscale images for {channel} channel ==================================")

            
            #Convert to png with a 0.1 resize for Ilastik
            convert_tiff_to_24bit_png_multiple_chan(input_tiff_path= os.path.join(img_folder,channel,filename), 
                                    output_png_path= os.path.join(ilastik_output,f'{filename.split(sep =".")[0]}'),
                                    resize = True,
                                    factor_resize= 5,
                                    brightness= False,
                                    brightness_factor= None,
                                    color = False,
                                    channel = channel)

Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0001.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0001.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0001_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0002.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0002.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0002_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0003.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0003.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0003_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0004.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0004.tif' to 'I:\Whole_brains\M2_

c:\Users\cartoa\AppData\Local\anaconda3\envs\data_science\lib\site-packages\PIL\Image.py:3402: DecompressionBombWarning: Image size (114475008 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0010.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0010.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0010_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0011.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0011.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0011_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0012.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0012.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0012_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0013.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0013.tif' to 'I:\Whole_brains\M2_

c:\Users\cartoa\AppData\Local\anaconda3\envs\data_science\lib\site-packages\PIL\Image.py:3402: DecompressionBombWarning: Image size (127270368 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0015.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0015.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0015_reduced' (24-bit PNG).


...

Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0019.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0019.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0019_reduced' (24-bit PNG).


c:\Users\cartoa\AppData\Local\anaconda3\envs\data_science\lib\site-packages\PIL\Image.py:3402: DecompressionBombWarning: Image size (121042176 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0020.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0020.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0020_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0021.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0021.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0021_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0022.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0022.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0022_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0023.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0023.tif' to 'I:\Whole_brains\M2_

c:\Users\cartoa\AppData\Local\anaconda3\envs\data_science\lib\site-packages\PIL\Image.py:3402: DecompressionBombWarning: Image size (141082656 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0027.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0027.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0027_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0028.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0028.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0028_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0029.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0029.tif' to 'I:\Whole_brains\M2_1\Stitched\images_24bits_QuickNII\M2_1_s0029_reduced' (24-bit PNG).
Converting I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0030.tif to (24-bit PNG).
Resizing by a factor: 20
Successfully converted 'I:\Whole_brains\M2_1\Stitched\CH3\M2_1_s0030.tif' to 'I:\Whole_brains\M2_

# **Rotating slices**


In [ ]:
to_rotate = ['0001','0002','0003','0004','0006','0024','0026','0027','0041','0043','0044','0047']
#pd.read_csv(r"D:\SFC\Images\TRAP_Ai14\B1_F3_SFC\to_flip.csv")

#to_rotate = to_rotate.iloc[:,0].values
file_list_QuickNII = [file for file in os.listdir(quickNII_output) if file.endswith(".png")]

import re

def extract_slice_number(filename):
    match = re.search(r'_s(\d+)\_reduced.png$', filename)
    return int(match.group(1)) if match else -1

file_list_sorted = sorted(file_list_QuickNII, key=extract_slice_number)
print(file_list_sorted)


for i, filename in enumerate(file_list_sorted):
    if f"{i+1:04d}" in to_rotate:
        image = Image.open(os.path.join(quickNII_output,filename))
        print(f'Rotating slice {i+1}')
        rotated_img = image.transpose(Image.FLIP_LEFT_RIGHT)
        save_path = os.path.join(quickNII_output, filename)
        print(f"Saving to: {save_path}")
        try:
            rotated_img.save(save_path)
            print(f"Image written {save_path}")
        except Exception as e:
            print(f"Error writing file: {e}")
    else:    
       continue


file_list_ilastik = [file for file in os.listdir(ilastik_output) if file.endswith(".png")]


def extract_slice_number(filename):
    match = re.search(r'_s(\d+)\_reduced.png$', filename)
    return int(match.group(1)) if match else -1

file_list_sorted = sorted(file_list_ilastik, key=extract_slice_number)
print(file_list_sorted)


for i, filename in enumerate(file_list_sorted):
    if f"{i+1:04d}" in to_rotate:
        image = Image.open(os.path.join(ilastik_output,filename))
        print(f'Rotating slice {i+1}')
        rotated_img = image.transpose(Image.FLIP_LEFT_RIGHT)
        save_path = os.path.join(ilastik_output, filename)
        print(f"Saving to: {save_path}")
        try:
            rotated_img.save(save_path)
            print(f"Image written {save_path}")
        except Exception as e:
            print(f"Error writing file: {e}")
    else:    
       continue


